# 00 - Problema ad hoc: por que necesitamos MCP

Objetivo: mostrar que varios wrappers para GitHub pueden parecer simples, pero terminan rompiendo contrato, permisos, auditoria y evolucion.


## Grafico guia

```mermaid
flowchart LR
    A["Cliente CLI"] --> D["Wrapper GitHub A"]
    B["Cliente Web"] --> E["Wrapper GitHub B"]
    C["Agente LLM"] --> F["Wrapper GitHub C"]
    D --> G["GitHub API"]
    E --> G
    F --> G
    G --> H["Problema: contratos dispersos"]
```


## Situacion docente

Tres clientes quieren crear repositorios y archivos en GitHub. Cada uno inventa su propia funcion, con nombres y payloads distintos. La API final puede ser la misma, pero el contrato que ve el LLM no es estable.


In [ ]:
def cli_create_repo(repo_name, readme_text):
    return {"source": "cli", "repo_name": repo_name, "readme_text": readme_text}

def web_new_repository(name, description, initialize):
    return {"source": "web", "name": name, "description": description, "initialize": initialize}

def agent_bootstrap_project(project, goal):
    return {"source": "agent", "project": project, "goal": goal}

examples = [
    cli_create_repo("aem4-demo", "README inicial"),
    web_new_repository("aem4-demo", "Demo MCP", True),
    agent_bootstrap_project("aem4-demo", "mostrar MCP"),
]
examples


## Lectura docente

Los tres caminos hablan de lo mismo, pero no hay un contrato unico. Si despues agregamos permisos, logs o versionado, cada wrapper lo resuelve distinto. MCP propone publicar capacidades con un contrato estable.


In [ ]:
mcp_contract = {
    "tool": "github_create_repo",
    "input": {
        "name": "string, min_length=3",
        "description": "string",
        "private": "boolean, default true",
    },
    "output": {
        "full_name": "owner/repo",
        "html_url": "url",
        "default_branch": "string",
    },
    "required_secret": "GITHUB_TOKEN",
    "audit": "github_mcp_audit_log.jsonl",
}
mcp_contract


## Pregunta al aula

Que error seria mas peligroso: que el wrapper falle con un mensaje claro, o que cree un repo publico cuando el usuario esperaba uno privado?
